In [1]:
import pandas as pd
import multiprocessing as mp
import numpy as np
import regex as re
import swifter
import html


/Users/emma/anaconda3/envs/macs30123/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = pd.read_parquet('/Users/emma/Desktop/thesis/actual_folder/clean/total_comments_B.parquet')

data

,body,author,score,id,parent_id,link_id,subreddit,author_flair_text,time
0,"yes its true, ron paul is anti-choice. anyone...",remembery,-7,c2gw9z,t3_2gw58,t3_2gw58,politics,None,2007-08-20 15:33:57
1,"news flash, two of them: 1. the republican pa...",captainhaddock,5,c2gwku,t3_2gw58,t3_2gw58,politics,None,2007-08-20 16:01:58
2,"i don't understand the argument citing ""a woma...",jk3us,2,c2gwkx,t3_2gw58,t3_2gw58,politics,None,2007-08-20 16:02:21
3,paul says all acts of violence are a state issue.,FrancisC,3,c2h1pb,t3_2gw58,t3_2gw58,politics,None,2007-08-20 22:28:09
4,"oh, i agree! but i think the world may not ye...",ayrnieu,1,c2hjs1,t1_c2gwkx,t3_2gw58,politics,None,2007-08-22 10:38:41
...,...,...,...,...,...,...,...,...,...
2486411,ive considered myself pretty moderate for a lo...,djc_tech,16,idkpeeg,t1_idk8xgw,t3_vjpnug,Conservative,va pede,2022-06-24 16:14:13
2486412,and the tv commercials were definitely a lie,Chefmark33,-4,iitp07w,t1_iitnqok,t3_wfcwn3,Conservative,2a,2022-08-03 19:56:46
2486413,you *might* be a worthless president of the un...,[deleted],1,ig779f7,t1_ig76sxa,t3_vzaflk,Conservative,None,2022-07-15 00:11:28
2486414,"i've met all these women, and they're not so d...",AdmiralAlluahAkbar,1,eo030rg,t3_bputar,t3_bputar,Conservative,None,2019-05-18 04:12:18


In [2]:
politics = pd.read_parquet("/Users/emma/Desktop/thesis/actual_folder/clean/politics_comments.parquet", engine="pyarrow")
# politics added an additional 153908 comments
# cons added 17651

In [3]:
conservative = pd.read_parquet("/Users/emma/Desktop/thesis/actual_folder/clean/conservative_comments.parquet")

In [4]:
total = pd.concat([politics, conservative], ignore_index=True)
print(total.shape)

total.head(5)

(2894950, 10)


,body,author,created_utc,score,id,parent_id,link_id,subreddit,author_flair_text,time
0,"Yes its true, ron paul is anti-choice. Anyone...",remembery,1187624037,-7,c2gw9z,t3_2gw58,t3_2gw58,politics,None,2007-08-20 15:33:57
1,"News flash, two of them:\r\n1. The Republican ...",captainhaddock,1187625718,5,c2gwku,t3_2gw58,t3_2gw58,politics,None,2007-08-20 16:01:58
2,"I don't understand the argument citing ""a woma...",jk3us,1187625741,2,c2gwkx,t3_2gw58,t3_2gw58,politics,None,2007-08-20 16:02:21
3,Paul says all acts of violence are a state issue.,FrancisC,1187648889,3,c2h1pb,t3_2gw58,t3_2gw58,politics,None,2007-08-20 22:28:09
4,[deleted],[deleted],1187664634,0,c2h3yl,t3_2gw58,t3_2gw58,politics,None,2007-08-21 02:50:34


In [37]:
2588680 - 2434772

306270 - 288619

17651

In [5]:
def to_dt(series):
    return pd.to_datetime(series, errors="coerce")

with mp.Pool(processes=4):  
    total["time"] = to_dt(total["time"])

In [7]:
def remove_urls(text):
    text = re.sub(r"(https?://\S+|www\.\S+|\S+\.(com|org|net)\S*)", "link", text)
    return text.strip()

total['body'] = total['body'].swifter.apply(remove_urls)

Pandas Apply: 100%|██████████| 2894950/2894950 [02:53<00:00, 16718.91it/s]


In [8]:
def clean_df(df, columns):
    def clean_text(text):
        if isinstance(text, str):
            text = re.sub(r'[^\x00-\x7F]+', '', text)  # non-ASCII
            text = html.unescape(text)  # Convert HTML entities
            text = re.sub(r':flag-[a-z]{2}:', '', text)  #flag emojis

            text = re.sub(r'^>.*(?:\n|$)', '', text, flags=re.MULTILINE) # quoted text

            # text = re.sub(r'[^\w\s.]', '', text)  # punctuation
            text = re.sub(r'\n', ' ', text)
            text = re.sub(r'\r', ' ', text)
            return text.strip().lower()  # strip spaces and lowercase
        return text

    df[columns] = df[columns].swifter.applymap(clean_text)
    return df

total = clean_df(total, ['body', 'author_flair_text'])

Pandas Apply: 100%|██████████| 5789900/5789900 [03:09<00:00, 30628.10it/s] 


In [14]:
pd.set_option('display.max_colwidth', None)

In [11]:
total.sample(10)

,body,author,created_utc,score,id,parent_id,link_id,subreddit,author_flair_text,time
1051090,"ah ok, thanks for the answer. is there any pre...",benjamoo,1630669561,1,hbfoigz,t1_hbd0lkx,t3_pghh91,politics,None,2021-09-03 11:46:01
2740427,the pro-abortion side will dodge this issue fo...,TripJammer,1516651711,0,dt2s6o3,t3_7s6518,t3_7s6518,Conservative,None,2018-01-22 20:08:31
1328654,"ahh yes, what democrats are best at /s",VanillaCupkake,1651597125,1,i76jkny,t1_i75srjl,t3_uhgdek,politics,None,2022-05-03 16:58:45
1921697,remember death panels?,SheeEttin,1657835310,11,ig6nhn0,t1_ig6gq6z,t3_vz2hmq,politics,america,2022-07-14 21:48:30
1491608,mr cruz replied he believed a liberal justices...,walter1950,1652111977,35,i7xmb39,t3_ulsurx,t3_ulsurx,politics,None,2022-05-09 15:59:37
2377677,"yeah, i keep waiting for that. like what part ...",WellWellWellthennow,1.693406767E9,1,jydqggm,t1_jyd9nal,t3_165d94o,politics,None,2023-08-30 14:46:07
446258,i'll believe it when i see it.,frodofappins,1486055364,1,dd8q1qg,t3_5rmin4,t3_5rmin4,politics,None,2017-02-02 17:09:24
183646,[huffington link,LinksToActualArticle,1355277892,2,c7f5hfn,t3_14ow61,t3_14ow61,politics,None,2012-12-12 02:04:52
141394,republicans!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!...,handburglar,1342714746,6,c5gbu0q,t1_c5gbqe9,t3_wta5b,politics,None,2012-07-19 16:19:06
1975289,the video at the top of the article shows her ...,isuckwithusernames,1658277486,1,igurjw6,t1_iguikqy,t3_w2zsxr,politics,None,2022-07-20 00:38:06


In [18]:
with mp.Pool(processes=4):
    total = total.drop('created_utc', axis=1)

In [19]:
# bots
bot_names = ['autotldr', 'automoderator', 'politicsmoderatorbot']

total = total[~total['author'].str.lower().isin(bot_names)]

In [16]:
total.shape

(2517543, 10)

In [20]:
del_rem = ['[deleted]', '[removed]']
total = total[~total['body'].isin(del_rem)]


In [29]:
total['author'].value_counts(normalize=True) * 100


author
[deleted]               3.026796
Carbonatite             0.159986
CarmineFields           0.106135
Long_Before_Sunrise     0.093325
rush42                  0.070395
                          ...   
LGBT_Beauregard         0.000040
CutiRomero              0.000040
Revenio                 0.000040
Dull-Huckleberry-693    0.000040
EddedTime               0.000040
Name: proportion, Length: 459008, dtype: float64

In [27]:
total[total['author'] == 'CarmineFields']

,body,author,created_utc,score,id,parent_id,link_id,subreddit,author_flair_text,time
315880,except republicans wont make those exceptions....,CarmineFields,1431650373,9,cr9i4e3,t1_cr9hejj,t3_35y7jy,politics,None,2015-05-15 00:39:33
315938,and laws designed to raise our maternal death ...,CarmineFields,1431660504,2,cr9nkm6,t1_cr9nf2e,t3_35y7jy,politics,None,2015-05-15 03:28:24
315941,are you willing to pay for social programs to ...,CarmineFields,1431660654,7,cr9nn8k,t1_cr9gp7w,t3_35y7jy,politics,None,2015-05-15 03:30:54
315950,and food stamps and welfare? you may well be ...,CarmineFields,1431661455,7,cr9o18c,t1_cr9nvcq,t3_35y7jy,politics,None,2015-05-15 03:44:15
316040,"so you want to force women to have babies, and...",CarmineFields,1431691704,1,cr9wxgs,t1_cr9ohl4,t3_35y7jy,politics,None,2015-05-15 12:08:24
...,...,...,...,...,...,...,...,...,...,...
2541585,there have been a lot more.,CarmineFields,1.726587251E9,5,lnl2foh,t3_1fiyrv6,t3_1fiyrv6,politics,None,2024-09-17 15:34:11
2541588,i never defend right-wing racists but in this ...,CarmineFields,1.726587392E9,160,lnl2vg4,t1_lnkin06,t3_1fiyrv6,politics,None,2024-09-17 15:36:32
2541590,i doubt it.,CarmineFields,1.726587473E9,3,lnl34mr,t1_lnkyxwj,t3_1fiyrv6,politics,None,2024-09-17 15:37:53
2541811,i guess i interpreted that they meant that her...,CarmineFields,1.726593751E9,2,lnlmvxr,t1_lnll2pp,t3_1fiyrv6,politics,None,2024-09-17 17:22:31


In [21]:
total = total[total['body'].str.len() > 0]

In [22]:
total.to_parquet('total_comments_B.parquet', index=False)

# comment stats

In [3]:
total = pd.read_parquet('total_comments.parquet')

In [42]:
print('total shape:', total.shape)
print('by subreddit:', total['subreddit'].value_counts())

total shape: (2489890, 10)
by subreddit: subreddit
politics        2278392
Conservative     211498
Name: count, dtype: int64


In [43]:
print('first comment:', total['time'].min())
print('last comment:', total['time'].max())


first comment: 2007-08-20 15:33:57
last comment: 2024-12-31 13:37:11


In [4]:
total['word_count'] = total['body'].swifter.apply(lambda x: len(x.split()))


Pandas Apply: 100%|██████████| 2489890/2489890 [00:15<00:00, 163073.31it/s]


In [6]:
pd.set_option('display.float_format', '{:.2f}'.format)
total['word_count'].describe()

count   2489890.00
mean         38.11
std          54.82
min           1.00
25%          11.00
50%          22.00
75%          44.00
max        2386.00
Name: word_count, dtype: float64

In [18]:
total['link_id'].nunique()

35439

In [17]:
print('mean comments:', len(total) / total['link_id'].nunique())
print('median:', total['link_id'].value_counts().median())



mean comments: 70.25847230452327
median: 10.0


In [9]:
total['body'].str.len().median()

122.0

In [7]:
total['word_count'].median()

22.0

In [ ]:
'''

subreddit
politics        2278455
Conservative     211769
Name: count, dtype: int64 first comment Timestamp('2007-08-20 15:33:57') 
last comment Timestamp('2024-12-31 13:37:11')

comment length:
count   2489890.00
mean         38.11
std          54.82
min           1.00
25%          11.00
50%          22.00
75%          44.00
max        2386.00
Name: body, dtype: float64


 '''